# Join Index Notebook Demo

The goal of this notebook is to demo the functionality of join index creation using [Aurum](https://github.com/mitdbg/aurum-datadiscovery). Creating a join index from a dataset repository consists of three main steps:

1) Create data summaries of data sources using DDProfiler, backed by Elasticsearch.

2) Build NetworkX data models from the data summaries, stored in Python pickle files.

3) Use the discovery API to build an index of joinable datasets, given a source dataset and max number of hops in the graph.

## Install requirements

In [1]:
pip install -r requirements.txt

  Using cached pandas-2.2.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (19 kB)
  Using cached bson-0.5.10-py3-none-any.whl
  Using cached alembic-0.8.5-py2.py3-none-any.whl
  Using cached amqp-1.4.9-py2.py3-none-any.whl.metadata (5.0 kB)
  Using cached appdirs-1.4.0-py2.py3-none-any.whl.metadata (8.2 kB)
  Using cached appnope-0.1.0-py2.py3-none-any.whl.metadata (685 bytes)
  Using cached arrow-1.3.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached backcall-0.2.0-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached billiard-3.3.0.22-py3-none-any.whl
  Using cached bitarray-3.1.1-cp311-cp311-macosx_11_0_arm64.whl.metadata (32 kB)
  Using cached bleach-2.1.4-py2.py3-none-any.whl.metadata (16 kB)
  Using cached certifi-2025.1.31-py3-none-any.whl.metadata (2.5 kB)
  Using cached charset_normalizer-3.4.1-cp311-cp311-macosx_10_9_universal2.whl.metadata (35 kB)
  Using cached click-6.6-py2.py3-none-any.whl.metadata (424 bytes)
  Using cached colorama-0.3.7-py2.py3-none-any.whl.metadata (13 

## Step 1: Create data summaries

### Deploy Elasticsearch

Running the profiler requires deploying a local instance of Elasticsearch, which can be downloaded [here](https://www.elastic.co/elasticsearch). Note that the currently supported version is 6.0.0.

Uncompress it and run from the root directory:

`./bin/elasticsearch`

This will start the server in `localhost:9200` by default.

### Configure DDProfiler

The profiler is built in Java, found under `seeker/src/aurum/ddprofiler`. The desired data sources to profile will be specified in a YAML file, e.g. a path to a folder with CSV files as in our example. The YAML file should follow the format specified in the `template.yml` in `ddprofiler/src/main/resources/template.yml`.

In [38]:
%cd aurum/ddprofiler/
! bash build.sh
! bash run.sh --sources ./src/main/resources/template_copy.yml

/Users/jiholee/SEEKER/seeker/src/aurum/ddprofiler


> Connecting to Daemon> IDLE<-------------> 0% INITIALIZING s]> Evaluating settings<-------------> 0% CONFIGURING s]> root project<-------------> 0% CONFIGURING s]<-------------> 0% EXECUTING s]> IDLE<=------------> 8% EXECUTING s]> :compileJava > Resolve dependencies of :compileClasspath<=------------> 8% EXECUTING s]<=------------> 8% EXECUTING s]<=------------> 8% EXECUTING s]<=------------> 8% EXECUTING s]> :compileJava<=------------> 8% EXECUTING s]<=------------> 8% EXECUTING [1s]<=------------> 8% EXECUTING [2s]
> Task :compileJava
/Users/jiholee/SEEKER/seeker/src/aurum/ddprofiler/src/main/java/core/Conductor.java:77: warning: [removal] Integer(int) in Integer has been deprecated and marked for removal
            String name = "Worker-" + new Integer(i).toString();
                                      ^


<=------------> 8% EXECUTING [2s]> :compileJava/Users/jiholee/SEEKER/seeker/src/aurum/ddprofiler/src/main/java/sources/imp

We can see that our data sources have been ingested into Elasticsearch and we are ready to build a network model from the profiled data sources.

## Step 2: Build model

The network builder constructs a NetworkX representation of our datasources and stores them in the path specified with the `--opath` flag in the form of Python pickle files.

In [39]:
%cd ..
! python networkbuildercoordinator.py --opath models/

/Users/jiholee/SEEKER/seeker/src/aurum
Building schema relation...
Building schema relation...OK
Total skeleton: 0.3242359161376953
!!1 0.3242359161376953
Time to TFIDF: 0.0025720596313476562
Time to create docs and TF-IDF: 
Create docs and TF-IDF: 0.0026540756225585938
Total index text: 0.012279987335205078
Create graph schema: 0.008625268936157227
Total schema-sim: 0.02383899688720703
!!2 0.02383899688720703
Total entity-sim: 0.0
Time to extract minhash signatures from store: 0.08196878433227539
!!3 0.08196878433227539
Total text-sig-sim (minhash): 0.22003507614135742
!!4 0.22003507614135742
Total num-sig-sim: 0.17365694046020508
!!5 0.17365694046020508
Total number PKFK: 757
Total PKFK: 0.004790067672729492
!!6 0.004790067672729492
Total time: 0.7467691898345947
!!7 0.7467691898345947
DONE!


When this step is complete, the following pickle files should be created in the specified models path:

- `content_sim_index.pkl`

- `graph.pickle`

- `id_info.pickle`

- `schema_sim_index.pkl`

- `table_ids.pickle`

## Step 3: Find join paths

Now, we can run our join path algorithm to find joinable datasets from a given data source. 

`join_path_api.py` accepts four parameters

1. data_path: path to the csv files
2. model_path: models generated by aurum which contains the graph information
3. query_table: the start table of all join paths.
4. max_hop: max hop of join paths. All join paths having the number of hops <= max_hop will be returned

In [41]:
! python join_path_api.py test_datasets/ ./models/ 2anc-iydk.csv 1

Finding join paths from 2anc-iydk.csv
source: 2anc-iydk.csv
# join paths: 52
2anc-iydk.of_students_in_grades_9_12 JOIN 2ay5-tqqe.of_students_in_grades_6_8
2anc-iydk.of_students_in_grades_9_12
datasource: 2anc-iydk.csv, unique_values: 52, non_empty_values: 52, total_values: 52, join_card: One-to-One, jaccard_similarity: 0, jaccard_containment: 0
2ay5-tqqe.of_students_in_grades_6_8
datasource: 2ay5-tqqe.csv, unique_values: 33, non_empty_values: 33, total_values: 33, join_card: One-to-One, jaccard_similarity: 0, jaccard_containment: 0
------------------------
2anc-iydk.of_students_in_grades_9_12 JOIN 2a5f-5ryi-2.multiple_race_categories
2anc-iydk.of_students_in_grades_9_12
datasource: 2anc-iydk.csv, unique_values: 52, non_empty_values: 52, total_values: 52, join_card: One-to-One, jaccard_similarity: 0, jaccard_containment: 0
2a5f-5ryi-2.multiple_race_categories
datasource: 2a5f-5ryi-2.csv, unique_values: 25, non_empty_values: 25, total_values: 25, join_card: One-to-One, jaccard_similarity